In [2]:
import polars as pl
from depersonal_lib.core.utils import MLPerformanceEvaluator
from depersonal_lib.core.transformations import EnteringIdentifiers
from depersonal_lib.core.transformations import generalize, suppress_rare_categories, drop_column
from depersonal_lib.core.privacy_models import KAnonymity

## Пример использования MLPerformanceEvaluator

In [ ]:
marital_dict = {'Married-AF-spouse': 'Previously-married', 
                'Married-spouse-absent': 'Previously-married', 
                'Widowed': 'Previously-married', 
                'Separated': 'Previously-married', 
                'Divorced': 'Previously-married', 
                'Married-civ-spouse': 'Married'}

In [ ]:
race_dict = {'Amer-Indian-Eskimo': 'Non-white', 
                'Asian-Pac-Islander': 'Non-white', 
                'Other': 'Non-white',
                'Black': 'Non-white'}

In [ ]:
data = pl.read_csv("adult.csv")
depersonalized_data = data.clone()

kan_orig = KAnonymity(['age', 'education', 'marital-status', 'race', 'gender','native-country'])
print('Уровень k-анонимности в исходных данных: ', kan_orig.analyze_k(data)[0])
print('Количество уязвимых групп : ',kan_orig.analyze_k(data)[1].shape[0])

depersonalized_data = drop_column(depersonalized_data, "education")
depersonalized_data = generalize(depersonalized_data, 'age', bins=[i for i in range(0,110,10)])
depersonalized_data = suppress_rare_categories(depersonalized_data, column="native-country", threshold=0.01)
depersonalized_data = depersonalized_data.with_columns(pl.col("marital-status").replace(marital_dict))
depersonalized_data = depersonalized_data.with_columns(pl.col("race").replace(race_dict))
transformer = EnteringIdentifiers("occupation")
depersonalized_data = transformer.transform(depersonalized_data)
transformer.save_mapping("adult_mapping.csv", format="csv")

kan_anon = KAnonymity(['age', 'marital-status', 'race', 'gender','native-country'])
min_k_anon, weak_anon = kan_anon.analyze_k(depersonalized_data)
print('Уровень k-анонимности в преобразованных данных:  ', min_k_anon)
print('Количество уязвимых групп : ', weak_anon.shape[0])

evaluator = MLPerformanceEvaluator(original_df=data, anonymized_df=depersonalized_data, target_column="income")
results = evaluator.evaluate_ml_performance(model_type="auto")

In [14]:
# Загрузка данных
data = pl.read_csv("adult.csv")
print("Исходные данные:")
data.head()

Исходные данные:


age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
25,"""Private""",226802,"""11th""",7,"""Never-married""","""Machine-op-inspct""","""Own-child""","""Black""","""Male""",0,0,40,"""United-States""","""<=50K"""
38,"""Private""",89814,"""HS-grad""",9,"""Married-civ-spouse""","""Farming-fishing""","""Husband""","""White""","""Male""",0,0,50,"""United-States""","""<=50K"""
28,"""Local-gov""",336951,"""Assoc-acdm""",12,"""Married-civ-spouse""","""Protective-serv""","""Husband""","""White""","""Male""",0,0,40,"""United-States""",""">50K"""
44,"""Private""",160323,"""Some-college""",10,"""Married-civ-spouse""","""Machine-op-inspct""","""Husband""","""Black""","""Male""",7688,0,40,"""United-States""",""">50K"""
18,"""?""",103497,"""Some-college""",10,"""Never-married""","""?""","""Own-child""","""White""","""Female""",0,0,30,"""United-States""","""<=50K"""


In [15]:
depersonalized_data = data.clone()

## Проверка K-анонимности на еще не обезличенном наборе данных

In [17]:
# Анализ анонимности по квази-идентификаторам
kan_orig = KAnonymity(['age', 'education', 'marital-status', 'race', 'gender','native-country'])
min_k_orig, weak_orig = kan_orig.analyze_k(data)
print('Текущий уровень k-анонимности: ', min_k_orig)

Текущий уровень k-анонимности:  1


In [18]:
print("\nСамые маленькие группы:")
weak_orig


Самые маленькие группы:


age,education,marital-status,race,gender,native-country,group_size
i64,str,str,str,str,str,u32
33,"""Assoc-acdm""","""Married-civ-spouse""","""Other""","""Male""","""Japan""",1
30,"""Some-college""","""Married-civ-spouse""","""Asian-Pac-Islander""","""Male""","""United-States""",1
57,"""Some-college""","""Widowed""","""Asian-Pac-Islander""","""Female""","""United-States""",1
49,"""Bachelors""","""Married-civ-spouse""","""Asian-Pac-Islander""","""Male""","""?""",1
32,"""Bachelors""","""Married-civ-spouse""","""White""","""Male""","""Japan""",1
…,…,…,…,…,…,…
22,"""Some-college""","""Married-civ-spouse""","""White""","""Male""","""Mexico""",1
35,"""Assoc-acdm""","""Divorced""","""White""","""Male""","""United-States""",1
43,"""Some-college""","""Divorced""","""Black""","""Female""","""?""",1


In [19]:
print("\nКоличество самых маленьких групп:")
print(weak_orig.shape[0])


Количество самых маленьких групп:
7065


## Удаление идентификатора  education

In [20]:
depersonalized_data = drop_column(depersonalized_data, "education")
depersonalized_data.head()

age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
i64,str,i64,i64,str,str,str,str,str,i64,i64,i64,str,str
25,"""Private""",226802,7,"""Never-married""","""Machine-op-inspct""","""Own-child""","""Black""","""Male""",0,0,40,"""United-States""","""<=50K"""
38,"""Private""",89814,9,"""Married-civ-spouse""","""Farming-fishing""","""Husband""","""White""","""Male""",0,0,50,"""United-States""","""<=50K"""
28,"""Local-gov""",336951,12,"""Married-civ-spouse""","""Protective-serv""","""Husband""","""White""","""Male""",0,0,40,"""United-States""",""">50K"""
44,"""Private""",160323,10,"""Married-civ-spouse""","""Machine-op-inspct""","""Husband""","""Black""","""Male""",7688,0,40,"""United-States""",""">50K"""
18,"""?""",103497,10,"""Never-married""","""?""","""Own-child""","""White""","""Female""",0,0,30,"""United-States""","""<=50K"""


## Обобщение атрибута age

In [21]:
# Обобщение возраста
depersonalized_data = generalize(depersonalized_data, 'age', bins=[i for i in range(0,110,10)])
depersonalized_data.head()

age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
str,str,i64,i64,str,str,str,str,str,i64,i64,i64,str,str
"""21-30""","""Private""",226802,7,"""Never-married""","""Machine-op-inspct""","""Own-child""","""Black""","""Male""",0,0,40,"""United-States""","""<=50K"""
"""31-40""","""Private""",89814,9,"""Married-civ-spouse""","""Farming-fishing""","""Husband""","""White""","""Male""",0,0,50,"""United-States""","""<=50K"""
"""21-30""","""Local-gov""",336951,12,"""Married-civ-spouse""","""Protective-serv""","""Husband""","""White""","""Male""",0,0,40,"""United-States""",""">50K"""
"""41-50""","""Private""",160323,10,"""Married-civ-spouse""","""Machine-op-inspct""","""Husband""","""Black""","""Male""",7688,0,40,"""United-States""",""">50K"""
"""11-20""","""?""",103497,10,"""Never-married""","""?""","""Own-child""","""White""","""Female""",0,0,30,"""United-States""","""<=50K"""


In [22]:
# Сколько раз встречается определенное значение возраста в оригинальном наборе данных
data['age'].value_counts().sort('count')

age,count
i64,u32
86,1
89,2
87,3
85,5
88,10
…,…
23,1329
33,1335
31,1335


In [23]:
# Сколько раз встречается определенное значение в обезличенном наборе
depersonalized_data['age'].value_counts().sort('count')

age,count
str,u32
"""81-90""",191
"""71-80""",759
"""61-70""",2781
"""11-20""",3685
"""51-60""",6229
"""41-50""",10419
"""21-30""",12193
"""31-40""",12851


## Подавление редких категорий

### Замена редкий значений в атрибуте native-country 

In [24]:
depersonalized_data['native-country'].value_counts().sort('count')

native-country,count
str,u32
"""Holand-Netherlands""",1
"""Hungary""",19
"""Honduras""",20
"""Scotland""",21
"""Yugoslavia""",23
…,…
"""Germany""",206
"""Philippines""",306
"""?""",935


In [25]:
# Подавление редких категорий в колонке 'native-country'
depersonalized_data = suppress_rare_categories(depersonalized_data, column="native-country", threshold=0.01)

In [26]:
depersonalized_data['native-country'].value_counts().sort('count')

native-country,count
str,u32
"""?""",935
"""Mexico""",1046
"""Other""",3270
"""United-States""",43857


### Преобразование значений в marital-status 

In [27]:
marital_dict = {'Married-AF-spouse': 'Previously-married', 
                'Married-spouse-absent': 'Previously-married', 
                'Widowed': 'Previously-married', 
                'Separated': 'Previously-married', 
                'Divorced': 'Previously-married', 
                'Married-civ-spouse': 'Married'}

In [29]:
depersonalized_data['marital-status'].value_counts().sort('count')

marital-status,count
str,u32
"""Married-AF-spouse""",37
"""Married-spouse-absent""",663
"""Separated""",1550
"""Widowed""",1560
"""Divorced""",6645
"""Never-married""",16178
"""Married-civ-spouse""",22475


In [30]:
depersonalized_data = depersonalized_data.with_columns(
    pl.col("marital-status").replace(marital_dict)
)

In [31]:
depersonalized_data['marital-status'].value_counts().sort('count')

marital-status,count
str,u32
"""Previously-married""",10455
"""Never-married""",16178
"""Married""",22475


### Изменение категорий в атрибуте race

In [32]:
depersonalized_data['race'].value_counts().sort('count')

race,count
str,u32
"""Other""",464
"""Amer-Indian-Eskimo""",481
"""Asian-Pac-Islander""",1575
"""Black""",4708
"""White""",41880


In [33]:
race_dict = {'Amer-Indian-Eskimo': 'Non-white', 
                'Asian-Pac-Islander': 'Non-white', 
                'Other': 'Non-white',
                'Black': 'Non-white'}

In [34]:
depersonalized_data = depersonalized_data.with_columns(
    pl.col("race").replace(race_dict)
)

In [35]:
depersonalized_data['race'].value_counts().sort('count')

race,count
str,u32
"""Non-white""",7228
"""White""",41880


## Введение идентификаторов для атрибута occupation

In [36]:
depersonalized_data['occupation'].value_counts().sort('count')

occupation,count
str,u32
"""Armed-Forces""",15
"""Priv-house-serv""",255
"""Protective-serv""",987
"""Tech-support""",1446
"""Farming-fishing""",1506
…,…
"""Sales""",5525
"""Adm-clerical""",5635
"""Exec-managerial""",6107


In [37]:
# Инициализируем преобразователь для колонки "occupation"
transformer = EnteringIdentifiers("occupation")

In [38]:
# Применяем трансформацию
depersonalized_data = transformer.transform(depersonalized_data)

In [39]:
#Какие значения теперь вместо исходных в колонке occupation (название преобразовано в вид id_***) 
depersonalized_data[depersonalized_data.columns[5]].value_counts().sort('count')

id_a71391,count
str,u32
"""8cb99d42af464ec3b286663635e1e0…",15
"""3f47506d17f8459f93de3cf90b0cf4…",255
"""5a370e63a5534dcb9a832efe24b794…",987
"""d16a8268bcc64119838950915749be…",1446
"""989da3d596144cee9e31d1c96ca32d…",1506
…,…
"""53f65911fd3e4a699e813968452914…",5525
"""565d6eff6ccd485e8a24a09bc635a7…",5635
"""45db32dd660e43dc834c1ffb14404f…",6107


In [26]:
# Сохраняем таблицу соответствий
transformer.save_mapping("adult_mapping.csv", format="csv")

In [27]:
# Какая таблица соответствий получилась (должна хранится отдельно от обезличенного набора данных)
mapping_tabel = pl.read_csv('adult_mapping.csv')
mapping_tabel

identifier,occupation,target_column
str,str,str
"""a1a96a9f62ec4328ac07efa5c49658…","""Tech-support""","""id_1a9376"""
"""486c28cba8cb4b0aaa710eeb5f76c8…","""Other-service""","""id_1a9376"""
"""2d16cbb4e73e4774bc4e35e245ac39…","""Adm-clerical""","""id_1a9376"""
"""78dfdd0c05694fc7800a4c777e4057…","""Transport-moving""","""id_1a9376"""
"""1fbb4a005d6f4076a6d679fb9fbc63…","""Sales""","""id_1a9376"""
…,…,…
"""3348ce3f1bf44804b01ec99e81ffd1…","""Machine-op-inspct""","""id_1a9376"""
"""ca76c440323b4ce68b1989bbb2ebc3…","""?""","""id_1a9376"""
"""3c0fc47a45fb4d20bac5447b0e5d99…","""Farming-fishing""","""id_1a9376"""


## Проверка K-анонимности 

In [40]:
# Анализ анонимности по квази-идентификаторам
kan_anon = KAnonymity(['age', 'marital-status', 'race', 'gender','native-country'])
min_k_anon, weak_anon = kan_anon.analyze_k(depersonalized_data)
print('Текущий уровень k-анонимности: ', min_k_anon)

Текущий уровень k-анонимности:  5


In [29]:
print("\nСамые маленькие группы:")
weak_anon


Самые маленькие группы:


age,marital-status,race,gender,native-country,group_size
str,str,str,str,str,u32
"""21-30""","""Previously-married""","""Non-white""","""Female""","""?""",5
"""61-70""","""Previously-married""","""White""","""Male""","""?""",5
"""11-20""","""Never-married""","""Non-white""","""Female""","""?""",5
"""41-50""","""Married""","""Non-white""","""Female""","""Mexico""",5
"""61-70""","""Previously-married""","""Non-white""","""Female""","""Mexico""",5
…,…,…,…,…,…
"""31-40""","""Married""","""Non-white""","""Female""","""Mexico""",5
"""81-90""","""Previously-married""","""White""","""Male""","""?""",5
"""41-50""","""Previously-married""","""Non-white""","""Female""","""Mexico""",5


## Оценка полезности

In [41]:
# Инициализация оценщика
evaluator = MLPerformanceEvaluator(
    original_df=data,
    anonymized_df=depersonalized_data,
    target_column="income"  # Целевая переменная
)

In [42]:
# Оценка производительности ML
results = evaluator.evaluate_ml_performance(model_type="auto")

In [43]:
results

{'model_type': 'classification',
 'test_size': 0.2,
 'random_state': 42,
 'metrics': {'accuracy_orig': 0.8575646507839544,
  'accuracy_anon': 0.8581755243331297,
  'precision_orig': 0.753862546616942,
  'precision_anon': 0.7598915989159891,
  'recall_orig': 0.6016156462585034,
  'recall_anon': 0.5960884353741497,
  'roc_auc_orig': 0.9093537984136091,
  'roc_auc_anon': 0.9111399680354069,
  'f1_orig': 0.8517600146708041,
  'f1_anon': 0.851936760339741,
  'accuracy_diff': 0.0006108735491753281,
  'precision_diff': 0.006029052299047155,
  'recall_diff': 0.005527210884353706,
  'roc_auc_diff': 0.001786169621797784,
  'f1_diff': 0.00017674566893688404}}

In [35]:
# Вывод результатов
print(f"Тип задачи: {results['model_type']}")
print("\nМетрики производительности:")
for metric, value in results["metrics"].items():
    if metric.endswith("_orig"):
        print(f"  Оригинальные данные ({metric.replace('_orig', '')}): {value:.4f}")
    elif metric.endswith("_anon"):
        print(f"    Обезличенные данные ({metric.replace('_anon', '')}): {value:.4f}")
    elif metric.endswith("_diff"):
        print(f"  Разница ({metric.replace('_diff', '')}): {value:.4f}")

Тип задачи: classification

Метрики производительности:
  Оригинальные данные (accuracy): 0.8576
    Обезличенные данные (accuracy): 0.8582
  Оригинальные данные (precision): 0.7539
    Обезличенные данные (precision): 0.7599
  Оригинальные данные (recall): 0.6016
    Обезличенные данные (recall): 0.5961
  Оригинальные данные (roc_auc): 0.9094
    Обезличенные данные (roc_auc): 0.9111
  Оригинальные данные (f1): 0.8518
    Обезличенные данные (f1): 0.8519
  Разница (accuracy): 0.0006
  Разница (precision): 0.0060
  Разница (recall): 0.0055
  Разница (roc_auc): 0.0018
  Разница (f1): 0.0002
